# **CODE-3:**

In [0]:
'''
CODE-3: OBTAINING CPCB DATAFRAME ACCORDING TO MYD04_3K TIMINGS EVERYDAY
==========================================================================================
TO DO/RELOOK:
-> Delete the first 16 lines of metadata in cpcb files 
-> Create CPCB filelist: ls *.csv > CPCBfilelist.txt
-> Careful in timings selected to shorten dataframe
-> Recheck the final datamatrix as sorting in ascending form is used for further codes 
-> Recheck column numbers wherever used

INPUT DATA      : Ground monitoring data
TIME PERIOD     : 01 Feb,2019 - 15 Mar,2019 and 01 Feb,2020 - 15 Mar,2020
OUTPUT          : chennai_ground_data_df

==========================================================================================
'''  

In [0]:
import os
from datetime import datetime
import pandas as pd
import numpy as np

To read all the cpcb files from a directory and impute missing timestamp during the time period

In [0]:
def ground_dataframe(filelist):
    #cpcb_df1 - stacking all cpcb excel files for all variables
    cpcb_df = pd.DataFrame() 
    for f in filelist:
        data = pd.concat(pd.read_excel(f, sheet_name=None))
        cpcb_tempdf = pd.DataFrame()
        cpcb_tempdf = cpcb_tempdf.append(data, ignore_index=True)
        #to get place's name for each sample
        cpcb_tempdf["place"] = f  
        cpcb_df = pd.concat([cpcb_df, cpcb_tempdf], axis=0)
    
    return cpcb_df

In [0]:
def impute_dates(work_df, files):
    imputed_df = pd.DataFrame()
    count=0

    work_df['From Date'] =  pd.to_datetime(work_df['From Date'])
    work_df['hour']  = pd.DatetimeIndex(work_df['From Date']).hour
    work_df['month']  = pd.DatetimeIndex(work_df['From Date']).month
    work_df['year']  = pd.DatetimeIndex(work_df['From Date']).year
    
    new_work_df = work_df[(work_df['hour'] > 11) &  (work_df['hour']< 16)]
    new_work_df = new_work_df[(new_work_df['month'] >=1) &  (new_work_df['month'] < 4)]
    
    for f in files:
        temp_df = new_work_df[(new_work_df['place']==f)]
        all_dates = pd.date_range("01-01-2016"+" 12:00", "31-03-2020"+" 15:45", freq="15min")
        comp_dates = list()
        for i in range(all_dates.shape[0]):
            comp_dates.append(all_dates[i].to_pydatetime())
        comp_dates = pd.DataFrame (comp_dates,columns=['From_Date'])
        comp_dates['hour']  = pd.DatetimeIndex(comp_dates['From_Date']).hour
        comp_dates['month']  = pd.DatetimeIndex(comp_dates['From_Date']).month
        comp_dates['year']  = pd.DatetimeIndex(comp_dates['From_Date']).year
        comp_dates = comp_dates[(comp_dates['hour'] > 11) &  (comp_dates['hour']< 16)]
        comp_dates = comp_dates[(comp_dates['month'] >=1) &  (comp_dates['month'] < 4)]
        new_dates = comp_dates.rename(columns={"From_Date": "From Date"})
        new_dates['PM2.5'] = None
        new_dates['RH'] = None
        new_dates['Temp'] = None

        for index, row in new_dates.iterrows():
            lst = temp_df[temp_df['From Date']==row['From Date']].index.tolist()
            if(len(lst)>0):
                new_dates.at[index, 'PM2.5'] = temp_df.at[lst[0], 'PM2.5']
                new_dates.at[index, 'Temp'] = temp_df.at[lst[0], 'Temp']
                new_dates.at[index, 'RH'] = temp_df.at[lst[0], 'RH']
        new_dates['place'] = f[:-5]
        new_dates = new_dates.drop(['hour', 'month', 'year'], axis=1)

        count = count+1
        print(count)
        
        imputed_df = pd.concat([imputed_df, new_dates], axis=0)
        
    return imputed_df

In [21]:
path = os.getcwd()
allfiles = os.listdir(path)
files_xlsx = [f for f in allfiles if (f[-5:] == '.xlsx')] 
ground_df = ground_dataframe(files_xlsx)
chennai_raw_ground_df = impute_dates(ground_df, files_xlsx)

1
2
3


In [22]:
chennai_raw_ground_df.to_csv('chennai_final_raw_ground_data7.csv')
print('dataframe created for all merged files')
chennai_ground_df = chennai_raw_ground_df.copy()

dataframe created for all merged files


In [0]:
#All the above files have already been processed, so read the csv file
chennai_ground_df = pd.read_csv('chennai_final_raw_ground_data.csv')

In [0]:
#create new columns and convert timestamp to individual numbers
chennai_ground_df['day']   = pd.DatetimeIndex(chennai_ground_df['From Date']).day
chennai_ground_df['month'] = pd.DatetimeIndex(chennai_ground_df['From Date']).month
chennai_ground_df['year']  = pd.DatetimeIndex(chennai_ground_df['From Date']).year
chennai_ground_df['hour']  = pd.DatetimeIndex(chennai_ground_df['From Date']).hour
chennai_ground_df['minute']= pd.DatetimeIndex(chennai_ground_df['From Date']).minute
#chennai_ground_df['second'] = pd.DatetimeIndex(chennai_ground_df['From Date']).second

In [24]:
#convert string to float and replace none values with 0
#df - dataframe, str - column to be changed 
def str_to_float(df, str):
    df[str].replace('None', 0, inplace = True)
    df[str].replace(np.nan, 0, inplace = True)
    df[str] = pd.to_numeric(df[str], downcast="float")
    return df
    
str_to_float(chennai_ground_df, 'PM2.5')
str_to_float(chennai_ground_df, 'RH')
str_to_float(chennai_ground_df, 'Temp')
#str_to_float(chennai_ground_df, 'WS')
#str_to_float(chennai_ground_df, 'WD')e

,From Date,PM2.5,RH,Temp,place,day,month,year,hour,minute
0,2016-01-01 12:00:00,78.550003,71.860001,29.250000,Alandur,1,1,2016,12,0
1,2016-01-01 12:15:00,66.550003,71.500000,29.020000,Alandur,1,1,2016,12,15
2,2016-01-01 12:30:00,39.270000,71.430000,28.969999,Alandur,1,1,2016,12,30
3,2016-01-01 12:45:00,35.820000,71.699997,29.150000,Alandur,1,1,2016,12,45
4,2016-01-01 13:00:00,27.980000,71.750000,29.180000,Alandur,1,1,2016,13,0
...,...,...,...,...,...,...,...,...,...,...
148907,2020-03-31 14:45:00,24.350000,55.400002,0.000000,Velachery,31,3,2020,14,45
148908,2020-03-31 15:00:00,25.610001,55.070000,0.000000,Velachery,31,3,2020,15,0
148909,2020-03-31 15:15:00,27.790001,56.080002,0.000000,Velachery,31,3,2020,15,15
148910,2020-03-31 15:30:00,27.790001,56.209999,0.000000,Velachery,31,3,2020,15,30


In [0]:
#sorting rows
chennai_ground_df  = chennai_ground_df.sort_values(['year','month','day','place','hour','minute'], ascending=[True, True, True, True, True, True])

In [0]:
#changing the index column
chennai_ground_df.reset_index(drop=True, inplace=True)

In [0]:
chennai_ground_df = chennai_ground_df.reindex(columns=['year','month','day','hour','minute','From Date','place','PM2.5','Temp','RH','PM10'])

In [28]:
chennai_ground_df.to_csv("chennai_final_ground_data7.csv")
print("File named 'chennai_ground_data' is created")

File named 'chennai_ground_data' is created
